# ARO 3D Studio — real TripoSR inference through the Studio (Colab)

**What this notebook proves when it succeeds:** real user images → the Studio's own upload/validation → persistent batch generation jobs → the **actual TripoSR model** (pinned revision, SHA-256-verified weights, safe loading) → raw assets → viewer preview → automatic preparation → validation → verified GLB export, for **three different object categories**, plus an invalid-file probe proving that one bad image does not stop its valid siblings.

**What it does not prove:** that outputs look good (a person must look and fill in the review), game-readiness, or anything about Unity 6. No mock, test double or procedural TITAN asset is used: the smoke script runs in `real_engine` mode and refuses to continue if the real engine is not runnable.

**Before you start:** Runtime ▸ Change runtime type ▸ **T4 GPU**. Run the cells **in order**, one at a time. Nothing here needs a secret or token (the model and the example images are public). If a cell fails, run the last cell ("Failure bundle") and send me the downloaded `aro-logs.zip` plus a screenshot of the failing cell.

## 1 · GPU and environment check
**Expected:** a table with `Tesla T4` (≈ 15 GB), `torch … cuda True`, free disk ≥ 20 GB. **If `cuda False`:** stop — the runtime is not a GPU runtime.

In [ ]:
import os, sys, platform, shutil
os.makedirs('/content/logs', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| cuda runtime', torch.version.cuda)
print('free disk GB:', round(shutil.disk_usage('/content').free/1e9, 1))
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then restart and run again'

## 2 · Get the Studio at a known revision
**Expected:** the last line prints the commit hash you are testing (`STUDIO_REF` below; `main` branch name or a commit). Keep that hash: it goes into the evidence. `pip` output is saved to `/content/logs/pip-studio.log`; on failure its tail is shown.

In [ ]:
STUDIO_REF = 'claude/zen-johnson-je3dzk'   # branch or full commit hash to verify
if not os.path.isdir('/content/aro'):
    !git clone https://github.com/AffanNamu/ARO-3D-STUDIO /content/aro
%cd /content/aro
!git fetch -q origin && git checkout -q $STUDIO_REF && git log -1 --format='STUDIO COMMIT %H  %s'
!pip install -q -r requirements.txt > /content/logs/pip-studio.log 2>&1 || (tail -40 /content/logs/pip-studio.log; false)
print('studio requirements installed')

## 3 · TripoSR source at the commit pinned in the lock, and its dependencies
**Expected:** `TSR COMMIT 107cefdc…` equals the lock's `code.commit`; `torchmcubes` compiles (**3–10 minutes**, quiet). `pip` output goes to `/content/logs/pip-triposr.log`.
The Studio refuses to load the model unless the weights match `aro_studio/engines/locks/triposr.json`, so this step only fetches code.

In [ ]:
import json
lock = json.load(open('aro_studio/engines/locks/triposr.json'))
TSR_COMMIT = lock['code']['commit']
if not os.path.isdir('/content/TripoSR'):
    !git clone -q https://github.com/VAST-AI-Research/TripoSR /content/TripoSR
!git -C /content/TripoSR checkout -q $TSR_COMMIT
head = !git -C /content/TripoSR rev-parse HEAD
print('TSR COMMIT', head[0]); assert head[0] == TSR_COMMIT, 'TripoSR checkout does not match the pinned commit'
!pip install -q -r requirements-triposr.txt > /content/logs/pip-triposr.log 2>&1 || (tail -60 /content/logs/pip-triposr.log; false)
os.environ['PYTHONPATH'] = '/content/TripoSR'
sys.path.insert(0, '/content/TripoSR')   # also for imports inside this kernel
print('triposr requirements installed')

## 4 · Preflight: the Studio's own availability report and weight verification
**Expected:** package versions (torch ≥ 2.6.0), `lock missing: []`, `availability ok: True`, then `OK: all files match the lock` after the ≈ 1.7 GB download at the pinned revision (`5b521936…`).
**If `availability ok: False`:** the printed `reasons` are the exact blocker — send them to me; do not edit the lock.

In [ ]:
import importlib, json, subprocess
for m in ('torch','transformers','omegaconf','einops','huggingface_hub','torchmcubes','tsr','trimesh','rembg','onnxruntime'):
    try:
        mod = importlib.import_module(m); print(f'{m:16s}', getattr(mod, '__version__', 'ok'))
    except Exception as e:
        print(f'{m:16s} IMPORT FAILED: {type(e).__name__}: {e}')
code_ = '''
import json
from aro_studio.engines.triposr import TripoSREngine
from aro_studio.engines.weights import load_lock
e = TripoSREngine(); av = e.availability("cuda")
print("lock missing:", load_lock("triposr").missing())
print("licence status:", e.info.license.status, "| model version:", e.info.version)
print("availability ok:", av.ok); print(json.dumps(av.to_dict(), indent=1))
'''
r = subprocess.run([sys.executable, '-c', code_], capture_output=True, text=True, env={**os.environ, 'PYTHONPATH': '/content/TripoSR:/content/aro'}); print(r.stdout, r.stderr[-2000:])
assert 'availability ok: True' in r.stdout, 'Engine not runnable: see the reasons above'
!python scripts/verify_triposr_weights.py 2>&1 | tail -12

## 5 · Khronos glTF validator (optional but recommended)
**Expected:** no output. Without it the export check still runs, but the Khronos result is recorded as "not run".

In [ ]:
!mkdir -p /content/gv && cd /content/gv && npm init -y >/dev/null 2>&1 && npm i gltf-validator >/dev/null 2>&1; ls /content/gv/node_modules | head -3
os.environ['ARO_GLTF_VALIDATOR_NODE_PATH'] = '/content/gv/node_modules'

## 6 · The test images (documented sources) and your vehicle photo
Three object categories are required — **vehicle**, **furniture/household**, **equipment/consumer product** — and each must be a different image.

| Category | Image | Source | Licence / provenance |
|---|---|---|---|
| furniture | `examples/chair.png` | TripoSR repository at the pinned commit | repository is MIT-licensed; authorship of the image itself is not stated upstream; SHA-256 pinned below |
| household object | `examples/teapot.png` | same | same |
| equipment / consumer product | `examples/robot.png` (toy-style robot) | same | same |
| **vehicle** | **your upload** | **you** | **you must state owner/licence below** — a photo you took, or a public-domain/CC0 photo with its URL |

For the vehicle use a clear, side or three-quarter view of one truck, bus or car (a plain or simple background gives the best result; cluttered backgrounds use neural matting, which downloads its own small model).

In [ ]:
from google.colab import files
up = files.upload()            # choose ONE vehicle photo (jpg/png/webp)
assert len(up) == 1, 'upload exactly one vehicle image'
VEHICLE_FILE = '/content/' + list(up)[0]

In [ ]:
VEHICLE_CATEGORY = 'heavy_truck'     # or: medium_truck, light_truck, bus, car, suv, pickup, ...
VEHICLE_NAME     = 'Vehicle photo'
VEHICLE_SOURCE   = ''                # REQUIRED, e.g. 'my own photo, taken 2026-10, all rights mine'  or  'Wikimedia Commons <URL>, CC0'
assert VEHICLE_SOURCE.strip(), 'State the owner/licence of the vehicle image in VEHICLE_SOURCE before continuing'

import hashlib, json
from PIL import Image
EX = '/content/TripoSR/examples'
EXPECTED = {   # SHA-256 of the exact files inspected before this notebook was written
  'chair.png':  '2503c12a74419d91a4c6c9f1affc48fee6e2b8b9091956ca6211e91ada57b5bf',
  'teapot.png': '66b0b8f7dd759e97d20665999885e0548f96c95dadee6bf9762ccd877c596b8c',
  'robot.png':  '092ee61f1729ac9ed516c8d031d071f44dfa6e79f6ece04f32c988721b2b657a',
}
for f, h in EXPECTED.items():
    got = hashlib.sha256(open(f'{EX}/{f}', 'rb').read()).hexdigest()
    assert got == h, f'{f} differs from the inspected file ({got})'
SRC = 'TripoSR repository examples/ at commit ' + TSR_COMMIT[:12] + ' (repo MIT-licensed; per-image authorship not stated upstream)'
items = [
  {'path': VEHICLE_FILE,          'category': VEHICLE_CATEGORY,   'name': VEHICLE_NAME, 'source': VEHICLE_SOURCE},
  {'path': f'{EX}/chair.png',     'category': 'furniture',         'name': 'Chair',      'source': SRC},
  {'path': f'{EX}/teapot.png',    'category': 'household_object',  'name': 'Teapot',     'source': SRC},
  {'path': f'{EX}/robot.png',     'category': 'equipment',         'name': 'Toy robot',  'source': SRC},
]
json.dump({'items': items}, open('/content/manifest.json', 'w'), indent=1)
from IPython.display import display
for it in items:
    im = Image.open(it['path']); t = im.copy(); t.thumbnail((220, 220))
    print(f"{it['name']:14s} {it['category']:17s} {im.size} {im.mode}  sha256 {hashlib.sha256(open(it['path'],'rb').read()).hexdigest()[:16]}…"); display(t)

## 7 · Run the real smoke test through the Studio
**Expected (takes ≈ 10–25 minutes the first time):** the Studio starts, the batch request returns `queued: 4, failed: 1` (the 5th file is a deliberately invalid text file and must be rejected while the four valid images keep going), jobs run one after another, and the JSON printed at the end shows `"mode": "real_engine"` and `"real_inference": "PASS"`. Exit code 0.
Per image the evidence records the model revision, device, input hash, job id, load/inference time, mesh statistics, output hash, preview, preparation steps, validation status and the re-opened GLB export.
**If it ends `BLOCKED`:** the engine was not runnable — the printed reasons are the blocker. **If `FAIL`/`PARTIAL`:** it still wrote `smoke-evidence.json`; send it with the logs.

In [ ]:
OUT = '/content/aro/docs/acceptance/evidence/ai/smoke'
!cd /content/aro && python scripts/real_inference_smoke.py --manifest /content/manifest.json --out $OUT --preset BALANCED 2>&1 | tee /content/logs/smoke-run.log | tail -45

## 8 · Check the evidence the way the acceptance report will
**Expected:** `status: PASS` and a line saying 4 different images went through the real engine; visual quality `PENDING_HUMAN_REVIEW`. The reader **rejects** test-double evidence, cache hits, missing hashes and fewer than 3 images.

In [ ]:
import subprocess, textwrap
code_ = textwrap.dedent('''
    from pathlib import Path
    from aro_studio.verification.evidence import read_smoke_evidence
    e = read_smoke_evidence(Path("/content/aro/docs/acceptance/evidence/ai/smoke/smoke-evidence.json"))
    print(e.status, "-", e.detail)
    for i in e.facts.get("items", []): print("  ", i)
''')
print(subprocess.run([sys.executable, '-c', code_], capture_output=True, text=True, env={**os.environ, 'PYTHONPATH': '/content/aro'}).stdout)
ev = json.load(open(OUT + '/smoke-evidence.json'))
print('mode:', ev['mode'], '| studio commit:', ev['studio_commit'], '| GPU:', (ev['environment'].get('cuda') or {}).get('device'))
for i in ev['items']:
    ri = i.get('real_inference', {}); pr = i.get('preparation', {}); ex = i.get('export', {})
    print(f"{i['name']:14s} {i.get('result','?'):5s} tris={ri.get('triangles')} load={ri.get('model_load_time_s')}s infer={ri.get('inference_time_s')}s device={ri.get('device')} cache_hit={ri.get('cache_hit')} prep={((pr.get('validation') or {}).get('status'))} export={ex.get('status')}")
print('invalid-file probe:', ev.get('negative_probe', {}).get('result'))

## 9 · LOOK at the results (this is the quality check)
Each pair is the **raw TripoSR mesh** (left) and the **prepared mesh** (right), from three angles. Judge: is it recognisable? Are parts missing, fused, hollow or flat? Which way is up and which way is the front?

In [ ]:
import glob
from PIL import Image
from IPython.display import display
for n, i in enumerate(ev['items'], 1):
    if not i.get('raw_views'): print(n, i['name'], '- no output'); continue
    print(f"=== {n:02d} {i['name']} ({i['category']}) ===")
    row = []
    for stage in ('raw', 'prepared'):
        for view in ('front', 'left', '3q'):
            p = f"{OUT}/views/{n:02d}-{stage}-{view}.png"
            if os.path.exists(p): row.append(Image.open(p).resize((300, 300)))
    sheet = Image.new('RGB', (300 * 3, 300 * 2), (230, 230, 230))
    for k, im in enumerate(row): sheet.paste(im, ((k % 3) * 300, (k // 3) * 300))
    display(sheet)

## 10 · Record your review (only after looking at step 9)
Fill **every** field for every image that produced a mesh — the script **refuses** empty fields, unknown names, or evidence from a test double, and never writes text for you. `recognisable` must be `yes`, `partially` or `no`. Be specific and honest: "back of the chair is a flat sheet" is the kind of thing worth recording.

In [ ]:
REVIEWER = ''     # your name
reviews = {
  VEHICLE_NAME:     {'recognisable': '', 'visual_defects': '', 'category_limitations': '', 'orientation_up_front': ''},
  'Chair':         {'recognisable': '', 'visual_defects': '', 'category_limitations': '', 'orientation_up_front': ''},
  'Teapot':        {'recognisable': '', 'visual_defects': '', 'category_limitations': '', 'orientation_up_front': ''},
  'Toy robot':     {'recognisable': '', 'visual_defects': '', 'category_limitations': '', 'orientation_up_front': ''},
}
json.dump(reviews, open('/content/review.json', 'w'), indent=1)
!cd /content/aro && python scripts/fill_human_review.py --evidence $OUT/smoke-evidence.json --reviews /content/review.json --reviewer "$REVIEWER"

## 11 · Package the evidence
**Expected:** two zips are downloaded.
* `smoke-evidence-repo.zip` — JSON, report, original input images, renders, prepared GLBs: **safe to commit** under `docs/acceptance/evidence/ai/smoke/`.
* `smoke-raw-meshes.zip` — the untouched raw TripoSR meshes (large): **keep it (Drive), do not commit**; hashes are in the JSON.
Send me `smoke-evidence-repo.zip` (and `aro-logs.zip` if anything failed).

In [ ]:
!cd /content/aro && python scripts/package_smoke_evidence.py --dir $OUT --out /content/package
from google.colab import files
files.download('/content/package/smoke-evidence-repo.zip'); files.download('/content/package/smoke-raw-meshes.zip')

## 12 · Failure bundle (run this if ANY cell failed)
Bundles every log and the evidence JSON (no weights, no secrets) so I can diagnose without guessing.

In [ ]:
!cd /content && zip -qr aro-logs.zip logs aro/docs/acceptance/evidence/ai/smoke/smoke-evidence.json aro/docs/acceptance/evidence/ai/smoke/smoke-report.md 2>/dev/null; ls -l aro-logs.zip
from google.colab import files; files.download('/content/aro-logs.zip')